1. Import Libraries

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, Dropout

2. Load Your Dataset

In [2]:
df = pd.read_csv("clean_text.csv")


texts = df["text"]
labels = df["label"]

3. Basic Info

In [3]:
print(df.head())
print("Unique labels:", np.unique(labels))

                                                text  label
0      i just feel really helpless and heavy hearted      4
1  ive enjoyed being able to slouch about relax a...      0
2  i gave up my internship with the dmrg and am f...      4
3                         i dont know i feel so lost      0
4  i am a kindergarten teacher and i am thoroughl...      4
Unique labels: [0 1 2 3 4 5]


4. Train/Test Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

5. Tokenization

In [ ]:
max_words = 10000
max_len = 100
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

6. Build Bi-LSTM Model

In [6]:
model = Sequential([
    Embedding(max_words, 128, input_length=max_len),
    Bidirectional(LSTM(128, return_sequences=True)),
    Bidirectional(LSTM(64)),
    Dropout(0.5),           # Add dropout
    Dense(64, activation='relu'),
    Dropout(0.3),           # Add dropout
    Dense(6, activation='softmax')
])

c:\Users\PC\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


7. Compile

In [7]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

8. Train

In [ ]:
model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test_pad, y_test)
)

Epoch 1/10
9846/9846 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.8575 - loss: 0.3800

In [ ]:
# Save the trained model and tokenizer for sharing
model.save('mood_model.h5')

import pickle
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

9.  MAIN FEATURE (PERCENTAGE OUTPUT)

In [ ]:
def predict_emotion(text):
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len)

    pred = model.predict(padded)

    # Negation handling
    negation_words = ["not", "never", "no", "don't", "can't", "isn't", "aren't", "wasn't", "weren't", "haven't", "hasn't", "won't", "wouldn't", "shouldn't", "couldn't", "didn't"]
    has_negation = any(word in text.lower() for word in negation_words)
    if has_negation:
        # Swap paired emotions to handle negation
        pred[0, [0,1]] = pred[0, [1,0]]  # swap sadness and joy
        pred[0, [2,3]] = pred[0, [3,2]]  # swap love and anger
        pred[0, [4,5]] = pred[0, [5,4]]  # swap fear and surprise

    # 🔥 Temperature scaling function (inside)
    def apply_temperature(probs, temperature=2.0):
        probs = np.log(probs + 1e-9) / temperature
        exp = np.exp(probs)
        return exp / np.sum(exp)

    # apply scaling
    scaled = apply_temperature(pred[0], temperature=2.0)
    percentages = scaled * 100

    emotion_map = {
        0: "sadness",
        1: "joy",
        2: "love",
        3: "anger",
        4: "fear",
        5: "surprise"
    }

    print("\n🎭 Emotion Distribution:\n")

    # Show all emotions
    for i in range(len(percentages)):
        print(f"{emotion_map[i]:10}: {percentages[i]:6.2f}%")

    # 🔥 Top 3 emotions
    top_indices = percentages.argsort()[-3:][::-1]

    print("\n🔥 Top Emotions:")
    for i in top_indices:
        print(f"{emotion_map[i]} ({percentages[i]:.2f}%)")

10. Test It

In [ ]:
predict_emotion("I am not happy with this product, it broke after one use.")

NameError: name 'predict_emotion' is not defined

In [ ]:
predict_emotion("Our system not only predicts the main emotion from user input but also shows how strongly each emotion is present using percentages. This helps simulate real human feelings, where a person can feel more than one emotion at the same time. To achieve this, we adjusted the model output using a technique called temperature scaling, which prevents the model from being too confident in one result. As a result, users can see a more balanced and meaningful emotional analysis instead of just a single label.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step

🎭 Emotion Distribution:

sadness   :  99.82%
joy       :   0.05%
love      :   0.09%
anger     :   0.01%
fear      :   0.00%
surprise  :   0.03%

🔥 Top Emotions:
sadness (99.82%)
love (0.09%)
joy (0.05%)


In [ ]:
# --- Data Enhancement Step ---
import pandas as pd
import numpy as np
import random

df = pd.read_csv('clean_text.csv')
print(f"Original dataset: {len(df)} samples")

Original dataset: 393822 samples


In [ ]:
negation_patterns = {
    'joy': [("not happy", 0), ("not joyful", 0), ("not excited", 4), ("never happy", 0)],
    'sadness': [("not sad", 1), ("not unhappy", 1), ("never sad", 1)],
    'anger': [("not angry", 1), ("never angry", 1), ("not mad", 1)],
    'fear': [("not afraid", 1), ("not scared", 1), ("never afraid", 1), ("no fear", 1)],
    'love': [("not love", 3), ("don't love", 3), ("hate", 3)],
    'surprise': [("not surprised", 1)]
}

In [ ]:
emotion_map = {
    0: 'sadness',
    1: 'joy',
    2: 'love',
    3: 'anger',
    4: 'fear',
    5: 'surprise'
}

In [ ]:
new_examples = []

for original_label in range(6):
    original_emotion = emotion_map[original_label]
    samples = df[df['label'] == original_label]['text'].tolist()

    if original_emotion in negation_patterns:
        for negation_word, new_label in negation_patterns[original_emotion]:
            for _ in range(500):
                if samples:
                    negated_text = f"I am {negation_word}"
                    new_examples.append({
                        'text': negated_text,
                        'label': new_label
                    })

new_df = pd.DataFrame(new_examples)
print(f"Generated {len(new_df)} negation examples")

Generated 9000 negation examples


In [ ]:
enhanced_df = pd.concat([df, new_df], ignore_index=True)
enhanced_df = enhanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Enhanced dataset: {len(enhanced_df)} samples")

Enhanced dataset: 402822 samples


In [ ]:
print(enhanced_df['label'].value_counts().sort_index())

label
0    120011
1    140530
2     29468
3     56277
4     44129
5     12407
Name: count, dtype: int64


In [ ]:
enhanced_df.to_csv('enhanced_text.csv', index=False)
print("Saved to enhanced_text.csv")

Saved to enhanced_text.csv
